# Implementation of the equations that make up the GSW functions for calculating the seawater surface density. A summary of the necessary steps are given in the cell below. 

# Mathematical equations

The **main equation** used is: 

$$
\hat{v} (S_A \Theta, p ) = v_u \sum_{i,j,k} v_{ijk} s^i \tau^j \pi^k  \tag{1}
$$

Where the values associated with i,j,k are given in the TEOS10 manual in Appendix K. 

$$
s = \sqrt{\frac{S_A + 24 gkg^-1}{S_{A_u}}}, \quad S_{A_u} = \frac{40 \times 35.16504 g kg^{-1}}{35}
$$

$$
\tau = \frac{\Theta}{\Theta_u}, \quad \Theta_u = 40^{\circ} C  
$$

$$
\pi = \frac{p}{p_u}, \quad P_u = 10^4 dbar
$$

$$
v_u = 1m^{3} kg^{-1}, \quad 
$$

However, p = 0 in this function and therefore $\pi \rightarrow 0$ when $k > 0$. Thats why the GSW function only needs salinity and temperature, not pressure for calculating the surface density: 

$$\hat{v} (S_A, \Theta, p = 0)$$


**Furthermore**, we also need to calculate the conserved temperature to use in the main equation. The following related equations are dedicated to this process.

For calculating the conserved temperature, we need to calculate both the entropy and a potential entalpi! The following steps are therefore needed in the calculations:

1. Calculation of the entropy (equation 2.10.1 from the TEOS manual): 

$$
\eta = \eta(S_A, t, p) = -g_T = - \frac{\partial g_T}{\partial T} \tag{2}
$$

In TEOS - empirical values are already used to create polynomial functions of $\eta$ - and I will rather use these estimated values directly with the aimn to reduce the need for computational calculations. These approximated values are found in appendix B of the TEOS and Copernicus manual! https://os.copernicus.org/articles/19/1719/2023/os-19-1719-2023.pdf

2. An iterativ calculation of potential temperature for the surface - ie. $\theta_0$. This is calculated using the Newton-Raphson iterative technique as illustrated in the equation below, found from TEOS10. It can also be estimated as an integral of the adiabatic lapse rate (Fofonoff, 1962 \& 1985), but I also suspect this would require potentially unneseccary computational power. 

$$
\eta (S_A, \theta, p_r) = \eta (S_A, t, p) \tag{3}
$$

The Newton Raphson iterative process is:
$$
x_{n+1} = x_n - \frac{f(x_n)}{f'(x_n)} \tag{4}
$$

Where we have to differentiate $g_t$ in our calculations. Furthermore, the full thermodynamic equation of entropy is given as: 

$$
\hat{\eta} (S_A, \Theta) = c_p^0 ln(1 + \Theta / T_0) + \alpha (\frac{S_A}{S_{SO}}) ln (\frac{S_A}{S_{SO}}) + P \{ 8,8 \} (s, \tau) \tag{5}
$$

Where: 

$$
T_0 = 273.15, \quad c_p^0 = 3991.86795711963 , \quad \alpha = - 9.309495003228781 , \quad S_{SO} = 35.16504 
$$

and the polynomial is differentiated such that the potency of $\tau$ \& $s$ is subtracted according to standard differentiation rules. 

3. Then we have to calculate the potential entalpy, where the reference pressure is always set to be zero because most heat flux activity is near the sea-surface. The reference pressure $p_r = 0$ dbar. This is calculated by using: 

$$
h^0 (S_A, t, p) = h (S_A, \theta, 0) = g(S_A, \theta, 0) - (T_0 + \theta)g_T (S_A, \theta, 0) \tag{6}
$$ 

Shortened to (I think - this is my doing hehe):
$$
h^0 = G - T * g_t 
$$

4. And yuhu now we can finally calculate the conserved temperature! Again with the use of the TEOS equations: 

$$
\Theta (S_A, t, p) = \tilde{\Theta} (S_A, \theta) = \frac{\tilde{h^0}(S_A, \theta)}{c_p^0} \tag{7}

$$

In [38]:
import numpy as np

In [53]:
def Gibbs(salinity, temperature):
    #The Gibbs empirical values gathered from Copernicus - for easier calculations of the Gibbs polynomials
    #The polynomials reach a highest power of 8
    
    """
    Constants
    """
    p_8 = {
    (0,0) : - 3.7102436569e-01,
    (1,0) :  3.0834502223e-04  , 
    (2,0) : - 3.2916987818e+00, 
    (3,0) :  7.2818259040e+00 , 
    (4,0) : - 5.6657256773e+00, 
    (5,0) :  2.8402903938e+00 , 
    (6,0) : - 8.9615123138e-01, 
    (7,0) : 1.0035964794e-01  ,
    (8,0) : 1.8140964105e-03  ,
    (0,1) : 3.0779211774e-02  ,
    (1,1) : 1.5006196848e-03  ,
    (2,1) : 1.2029316021e-01  ,
    (3,1) : 3.7464975805e-01  ,
    (4,1) : - 6.0590428227e-01,
    (5,1) : 6.4365865093e-02  ,
    (6,1) : 2.4626795446e-02  ,
    (7,1) : - 1.0335853091e-02 ,
    (0,2) : 2.3045093877e+00 ,
    (1,2) : - 5.4154968624e-03 ,
    (2,2) : - 2.5098282844e+00,
    (3,2) : 1.9163697628e-02,
    (4,2) : 9.6230320461e-02,
    (5,2) : 3.7953034101e-02,
    (6,2) : - 5.1206778774e-04,

    (0,3) : - 8.4974032876e-01 ,
    (1,3) : - 1.3727475447e-02,
    (2,3) : 8.6969911602e-01,
    (3,3) : 1.1127539375e-01,
    (4,3) : - 8.7616123860e-02,
    (5,3) : - 1.6250024449e-02,
    (0,4) : 4.1807750439e-01,
    (1,4) : 5.1388181100e-02,
    (2,4) : - 3.1917000611e-01,
    (3,4) : - 4.4999965986e-02,
    (4,4) : 3.3822211876e-02,
    (0,5) : - 1.9191736060e-01,
    (1,5) : - 5.3890029514e-02,
    (2,5) : 9.3472917957e-02,
    (3,5) : - 4.9779616704e-04,
    (0,6) : 6.6066546976e-02,
    (1,6) : 2.4144978278e-02,
    (2,6) : -1.2850921670e-02,
    (0,7) : -1.3678360946e-02,
    (1,7) : -4.1337102429e-03,
    (0,8) : 1.1180283076e-03}

    #Then we calculate the polynomial values where the potency is differentiated one time and included in the values
    theta0 = 40.0 #given in article of thermodynamic potential in seawater 
    T0 = 273.15
    SSO = 35.16504

    tau = (T0  + temperature) / theta0
    s = np.sqrt(salinity / SSO)
    
    gibbs_non_derivative = 0.0
    gibbs_poly_derivative = 0.0
    for (i,j), coeff in p_8.items():
        gibbs_non_derivative += coeff * (s**i) * (tau**j) 
        if j > 0:
            gibbs_poly_derivative += j * coeff * (s **i) * (tau**(j-1))
    #Now because we are differentiating based on  T (which is what we have) - we will need to scale our tau to get the non-dimensional right var
    gibbs_poly_derivative_scaled = gibbs_poly_derivative / theta0
    gibbs_non_derivative_scaled = gibbs_non_derivative / theta0

    return gibbs_non_derivative_scaled, gibbs_poly_derivative_scaled

In [54]:
tau = 15
salinity = 35
result = Gibbs(tau, salinity)
print(result)

%pip install gsw 
import gsw

(np.float64(-17.043480758058603), np.float64(1.7325402999255402))

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [55]:
import numpy as np 
from scipy.differentiate import derivative

In [56]:
"""THIS IS A TEST CELL TO MAKE SURE THE FUNCTIONS MATCH THE GSW FUNCTIONS"""

theta0 = 40.0
sso = 35.16504

CT = tau * theta0  #Conservative temperature [°C]
SA = salinity**2 * sso  #Absolute salinity [g/kg]

#Entropy from GSW
entropy_gsw = gsw.entropy_from_CT(SA, CT)

specific_heat = 3991.86795711963  # J/kg/K
alpha = -9.309495003228781  # J/kg/K
T0 = 273.15  # Kelvin
log_term1 = specific_heat * np.log(1 + CT / T0)
log_term2 = alpha * (SA / sso) * np.log(SA / sso)

#Gibbs polynomial calculations
gibbs_non_derivative, gibbs_derivative = Gibbs(salinity, tau)

#analytical entrophy 
eta_analytical = log_term1 + log_term2 + gibbs_derivative

print(f"GSW solution: {entropy_gsw:.6f} J/kg/K")
print(f"Analytical Solution: {eta_analytical:.6f} J/kg/K")

GSW solution: -693014441744502.625000 J/kg/K
Analytical Solution: -76483.839804 J/kg/K


We have to differentiate the logarthitmic expression also. The second term $\rightarrow 0$, because it is independent of $\Theta$. So we are left with:

$$

\frac{\partial}{\partial \Theta} (c_p \cdot ln(1+ \frac{\Theta}{T_0})) =
(\frac{c_p}{T_0 + \Theta})'
$$

In [43]:
def iterative_potential_temp(salinity, temperature, pressure = None):
    """
    Definition: 
    Calculating the iterative potential temperature. Eta varies with absolute salinity and temperature,
    so we differentiate the gibbson poly sum with respect to the temperature. 
    
    OBS! This iterative process does not yet work for other pressure levels than 0dbar. 
    This function will therefore need to be changed when we start including the other layers
    and p no longer is equal to zero.
    """
    theta = temperature #we will use temperature as a starting value
    tolerance = 1e-14 #as stated in potential temperature section 3.1 from IOC et al 2010
    max_iterations = 100 

    specific_heat = 3991.86795711963  # J/kg/K
    alpha = -9.309495003228781  # J/kg/K
    T0 = 273.15  # Kelvin
    SSO = 35.16504  # Standard Ocean Salinity

    if pressure is not None: 
        for i in range(max_iterations):
            log_term1 = specific_heat * np.log(1 + theta / T0)
            log_term2 = alpha * (SA / SSO) * np.log(SA / SSO)
            log_term1_der = specific_heat / (T0 + theta)

            gibbs_non_derivative, gibbs_scaled_derivative = Gibbs(theta, salinity)
            eta_t_non = log_term1 + log_term2 + gibbs_non_derivative
            eta_t_der = log_term1_der + gibbs_scaled_derivative  

            #The Newton Raphson iteration
            delta_theta = -1 * (eta_t_non / eta_t_der)
            theta_new = theta + delta_theta 

            #convergence test
            # Sjekk for konvergens
            if abs(theta_new - theta) < tolerance:
                print(rf'Approximated solution is found after {i+1} of iterations: {theta_new} .')
                return theta_new

            # Oppdater theta
            theta = theta_new
        print(r'Conergence was not reached and a solution for \theta is not found.')
    if pressure is None:
        theta = temperature 
        print(f'The potential temperature is equal to given temperature at the ocean surface: {theta}.')

    return theta

In [44]:
#####TEST#####
theta = iterative_potential_temp(salinity, tau)
print( gsw.pt0_from_t(salinity, tau, 0))

The potential temperature is equal to given temperature at the ocean surface: 15.
15.0



Potential entalpy

reference pressure = 0dbar = 1013hPa

Remember that $g_T$ is the Gibbs temperature derivated function, while $g = $ the non-derivated function.

$\eta = -g_T$

In [45]:
def potential_entalpy_conservative_temperature(salinity, temperature, pressure = None):
    
    #define constants
    T0 = 273.15 #Kelvin
    specific_heat = 3991.86795711963  # J/kg/K

    #calculate the essentials 
    theta = iterative_potential_temp(salinity, temperature, pressure = pressure)
    g, g_t = Gibbs(salinity, theta)


    #for the sigma0 function - we will only use the p = 0 case, and this function must be developed
    #further when working with more than one layer to include the pressure effects
    h0 = g + ((T0 + theta) * g_t)

    conserved_temperature = h0 / specific_heat 
    return conserved_temperature

In [46]:
c_t = potential_entalpy_conservative_temperature(salinity, tau, None)
print(c_t)    

The potential temperature is equal to given temperature at the ocean surface: 15.
-2.2781505646395184


In [47]:
#test against the gsw 
CT_gsw = gsw.CT_from_pt(SA, theta)

In [48]:
CT_gsw

np.float64(-450709411.1202886)